# Approach 4 — Bézier Regression from Real Images

**Before running:** Runtime → Change runtime type → **T4 GPU**

### What this trains
`HandwritingCNN`: CNN takes a real handwriting image → predicts 6 cubic Bézier curves  
(6 curves × 4 control points × 2 coordinates = 48 values).  
Loss: MSE on normalised control-point coordinates.  
At inference: pick any real reference image → predict curves → render.

### First run note
The first training run skeletonises all 2000+ handwriting images to extract Bézier labels.  
This builds `bezier_labels.npy` (cache) and takes ~5 minutes. Subsequent runs load from cache.

### After training
Download `style_net.pt` and place it at:  
`approaches/04_BezierRegression_RealImages/checkpoints/style_net.pt`  
Then run locally: `cd approaches/04_BezierRegression_RealImages && python run.py`

In [ ]:
# Cell 1: Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU — go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3: Install dependencies
!pip install scipy scikit-image -q
print('Dependencies installed')

In [ ]:
# Cell 4: Clone repo
import os

REPO      = 'AI-Powered-Handwriting-Generation-System'
REPO_PATH = f'/content/{REPO}'

if os.path.exists(REPO_PATH):
    print('Repo exists — pulling latest...')
    os.system(f'git -C {REPO_PATH} pull')
else:
    print('Cloning repo...')
    os.system(f'git clone --depth 1 https://github.com/Abdullahshaz70/{REPO}.git {REPO_PATH}')

APPROACH_DIR = f'{REPO_PATH}/approaches/04_BezierRegression_RealImages'
print('Approach dir:', APPROACH_DIR)

In [ ]:
# Cell 5: Configure training
EPOCHS = 100   # 100-150 recommended

print(f'Will train for {EPOCHS} epochs')
print('Note: first run builds bezier_labels.npy cache — takes ~5 minutes')

In [ ]:
# Cell 6: Train
os.chdir(APPROACH_DIR)
!python run.py --train --epochs {EPOCHS}

In [ ]:
# Cell 7: Save checkpoint to Drive + download
import shutil
from google.colab import files

ckpt_src  = f'{APPROACH_DIR}/checkpoints/style_net.pt'
drive_dir = '/content/drive/MyDrive/handwriting_checkpoints/04_BezierReg'
os.makedirs(drive_dir, exist_ok=True)

shutil.copy(ckpt_src, os.path.join(drive_dir, 'style_net.pt'))
print('Saved to Drive:', drive_dir)

files.download(ckpt_src)
print('\nDownload started.')
print('Place style_net.pt in:  approaches/04_BezierRegression_RealImages/checkpoints/')
print('Then run locally:       cd approaches/04_BezierRegression_RealImages && python run.py')

In [ ]:
# Cell 8: Preview — generate a-j for each writer
import sys, random
sys.path.insert(0, f'{APPROACH_DIR}/src')

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from model   import HandwritingCNN
from bezier  import label_to_curves, add_variation, draw_bezier
from data    import CHAR_TO_IDX, scale_to_canvas, CANVAS_SIZE

SKIP_DIRS = {'Writers_Zip', 'output_preview', '__pycache__'}
DATA_ROOT = f'{REPO_PATH}/Data/Writers_pngs'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt   = torch.load(f'{APPROACH_DIR}/checkpoints/style_net.pt', map_location=device)
model  = HandwritingCNN(num_chars=62).to(device)
model.load_state_dict(ckpt['model_state']); model.eval()

writer_names = ckpt['writer_names']
refs = {}
dirs = sorted([e for e in os.scandir(DATA_ROOT) if e.is_dir() and e.name not in SKIP_DIRS],
              key=lambda e: e.name)
for wid, entry in enumerate(dirs):
    refs[wid] = [Image.open(os.path.join(entry.path, f)).convert('L')
                 for f in sorted(os.listdir(entry.path)) if f.lower().endswith('.png')][:5]

def gen_char(char, writer_idx):
    ref = random.choice(refs[writer_idx])
    arr = np.array(ref.resize((CANVAS_SIZE, CANVAS_SIZE)))
    arr = scale_to_canvas(np.where(arr < 128, 0, 255).astype(np.uint8), target_fill=0.80)
    t   = torch.from_numpy(arr.astype(np.float32) / 127.5 - 1.0).unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad():
        label = model(t, torch.tensor([CHAR_TO_IDX[char]], device=device)).squeeze().cpu().numpy()
    curves = label_to_curves(label)
    canvas = np.ones((CANVAS_SIZE, CANVAS_SIZE), dtype=np.uint8) * 255
    return draw_bezier(canvas, add_variation(curves), thickness=3)

test_chars = list('abcdefghij')
fig, axes  = plt.subplots(len(writer_names), len(test_chars),
                           figsize=(len(test_chars)*1.5, len(writer_names)*1.5))
if len(writer_names) == 1: axes = [axes]

for wi in range(len(writer_names)):
    for ci, ch in enumerate(test_chars):
        axes[wi][ci].imshow(gen_char(ch, wi), cmap='gray', vmin=0, vmax=255)
        if wi == 0: axes[wi][ci].set_title(ch)
        if ci == 0: axes[wi][ci].set_ylabel(writer_names[wi].replace('writer_',''), fontsize=7)
        axes[wi][ci].axis('off')

plt.suptitle(f'Bézier Regression — {EPOCHS} epochs')
plt.tight_layout()
plt.show()